# 07. XGBoost 튜닝 실험 - 과적합 문제 발견 및 분석
> **⚠️ 이 노트북은 최종 모델이 아닙니다**
> 과적합 문제를 발견하고 원인을 분석한 탐색 과정입니다.
> 개선된 최종 모델은 `08_xgboost_improved.ipynb` → `09_peak_targeting.ipynb` 에 있습니다.

## 이 노트북의 역할
| 단계 | 내용 | 결과 |
|------|------|------|
| 1차 튜닝 | Optuna 기본 적용 | Train RMSE 41 vs Test RMSE 244 → **과적합 발견** |
| 2차 튜닝 | TimeSeriesSplit CV 추가 | CV RMSE 편차 77.9 → **불안정성 확인** |
| 3차 튜닝 | Early Stopping + L1/L2 규제 추가 | 큰 개선 없음 → **피처 엔지니어링 필요 결론** |
| 4차 실험 | 최소 피처(FEATURES_V1)로 재실험 | **08번 개선 방향 설정** |

## 핵심 발견
```
학습 RMSE : 41.0
테스트 RMSE: 244.8
과적합 갭 : 203.8  ← 이 숫자가 문제의 시작
```
단순히 파라미터 조정으로는 해결 안 됨 → Rolling 피처, 고온 피처, 가중치 전략 필요

## 📦 라이브러리 임포트
- `MinMaxScaler` : 이 단계에서 스케일링 시도했으나 XGBoost는 스케일 불필요 → 이후 제거
- 나머지는 08·09번과 동일한 구성

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import platform
import os
import optuna
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_percentage_error
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings("ignore")

## 🔤 폰트 설정
한글 폰트(Pretendard) 설정. 이후 모든 노트북에 동일하게 적용.

In [2]:
# 1. 폰트 파일 경로 설정
font_path = r"C:\Users\User\Desktop\MyRepo\Portfolio\Portfolio_ver2\5_fonts\Pretendard-Medium.otf"

# 2. 폰트 이름 가져오기
font_name = fm.FontProperties(fname = font_path).get_name()
fm.fontManager.addfont(font_path)

# 3. Matplotlib의 기본 폰트로 설정
plt.rc("font", family = font_name)

# 4. 마이너스 기호 깨짐 방지
plt.rcParams["axes.unicode_minus"] = False

print(f"설정된 폰트 이름: {font_name}")

설정된 폰트 이름: Pretendard


## 📂 데이터 로드
기본 로드. `sort_values()` 정렬이 빠진 상태 → 이후 08번에서 수정

In [3]:
df = pd.read_csv(r"C:\Users\User\Desktop\MyRepo\Portfolio\Portfolio_ver2\1_data\processed\df_final_v2.csv")
df["datetime"] = pd.to_datetime(df["datetime"])

## 🧹 결측치 처리
강수량·일사량 NaN을 0으로 채움. 물리적으로 타당한 처리.

In [4]:
df["datetime"] = pd.to_datetime(df["datetime"])
df["강수량(mm)"] = df["강수량(mm)"].fillna(0)
df["일사(MJ/m2)"] = df["일사(MJ/m2)"].fillna(0)

## 1. Optuna 하이퍼파라미터 튜닝 (프로토타입 → 고도화)

## 🔍 1차 Optuna 튜닝 - 기본 버전
**이 단계의 문제점**
- `max_depth` 탐색 범위가 3~10으로 너무 넓음 → 깊은 트리(max_depth=6) 선택 → 과적합 유발
- `min_child_weight` 범위 1~20으로 너무 낮음 → 작은 샘플에도 분기 허용 → 과적합
- L1/L2 규제 파라미터 없음 → 과적합 억제 수단 부재

**사용 피처 (FEATURES_TUNED, 21개)**
- 기상 6개 + 시간 6개 + 냉난방도일 2개 + lag 2개 + rolling 1개 + 체감지수 2개
- rolling_24h 하나만 있고 세분화된 rolling 없음 → 단기 패턴 학습 부족

In [5]:
# 1. 피처 정의
FEATURES_TUNED = [
    "기온(°C)", "강수량(mm)", "풍속(m/s)", "습도(%)", "일사(MJ/m2)", "전운량(10분위)",
    "미세먼지(PM10)", "초미세먼지(PM25)",
    "hour", "dayofweek", "month", "is_weekend", "is_holiday", "is_off",
    "CDD", "HDD", "lag_24", "lag_168", "rolling_24h",
    "체감온도", "불쾌지수"  
]

TARGET = "전력사용량(MWh)"

# 2. 학습 / 테스트 분리
train = df[df["datetime"].dt.year == 2023]
test = df[df["datetime"].dt.year == 2024]

X_train = train[FEATURES_TUNED]
y_train = train[TARGET]
X_test = test[FEATURES_TUNED]
y_test = test[TARGET]

# 3. Optuna 목적 함수
def objective(trial):
    params = {
        "n_estimators"     : trial.suggest_int("n_estimators", 100, 1000),
        "max_depth"        : trial.suggest_int("max_depth", 3, 10),
        "learning_rate"    : trial.suggest_float("learning_rate", 0.01, 0.3),
        "subsample"        : trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree" : trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight" : trial.suggest_int("min_child_weight", 1, 10),
        "random_state"     : 42,
        "n_jobs"           : -1
    }
    model = XGBRegressor(**params)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    return np.sqrt(mean_squared_error(y_test, pred))

# 4. Optuna 실행
study = optuna.create_study(direction = "minimize")
study.optimize(objective, n_trials = 100, show_progress_bar = True)

# 5. 최적 파라미터 출력
print(f"\n최적 RMSE: {study.best_value:.1f}")
print(f"최적 파라미터: {study.best_params}")

# 6. 최적 모델 재학습
best_model = XGBRegressor(**study.best_params, random_state = 42, n_jobs = -1)
best_model.fit(X_train, y_train)
pred = best_model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)
mape = mean_absolute_percentage_error(y_test, pred) * 100

print(f"\n고도화 모델 성능")
print(f"RMSE: {rmse:.1f}")
print(f"R2: {r2:.4f}")
print(f"MAPE: {mape:.2f}%")

# 7. 프로토타입 vS 고도화 비교
print(f"\n프로토타입 VS 고도화 비교")
print(f"{'':15s} | {'RMSE':>8} | {'R²':>8} | {'MAPE':>8}")
print("=" * 45)
print(f"{'프로토타입(Full)':15s} | {'255.4':>8} | {'0.9709':>8} | {'2.86%':>8}")
print(f"{'고도화(Tuned)':15s} | {rmse:>8.1f} | {r2:>8.4f} | {mape:>8.2f}%")

[I 2026-03-04 12:34:07,046] A new study created in memory with name: no-name-fc7dcb89-7dca-4914-b652-1cf985cc0cdd


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-03-04 12:34:07,765] Trial 0 finished with value: 295.8497427960687 and parameters: {'n_estimators': 447, 'max_depth': 7, 'learning_rate': 0.23842272134251888, 'subsample': 0.7399798388196304, 'colsample_bytree': 0.743789049007081, 'min_child_weight': 5}. Best is trial 0 with value: 295.8497427960687.
[I 2026-03-04 12:34:07,917] Trial 1 finished with value: 252.8936519937778 and parameters: {'n_estimators': 242, 'max_depth': 3, 'learning_rate': 0.19006009968092968, 'subsample': 0.7752959725535238, 'colsample_bytree': 0.9805019210240334, 'min_child_weight': 7}. Best is trial 1 with value: 252.8936519937778.
[I 2026-03-04 12:34:08,319] Trial 2 finished with value: 268.7873811212126 and parameters: {'n_estimators': 415, 'max_depth': 6, 'learning_rate': 0.29775053037630905, 'subsample': 0.963846911973468, 'colsample_bytree': 0.8925498716979787, 'min_child_weight': 7}. Best is trial 1 with value: 252.8936519937778.
[I 2026-03-04 12:34:09,132] Trial 3 finished with value: 262.51299074

In [6]:
# 5. 최적 파라미터 출력
print(f"\n최적 RMSE: {study.best_value:.1f}")
print(f"최적 파라미터: {study.best_params}")


최적 RMSE: 241.6
최적 파라미터: {'n_estimators': 828, 'max_depth': 7, 'learning_rate': 0.044455341535896184, 'subsample': 0.6238432312116119, 'colsample_bytree': 0.8148472177489084, 'min_child_weight': 10}


## 📊 1차 튜닝 결과
```
최적 RMSE  : 241.6
max_depth  : 7  ← 너무 깊음
n_estimators: 828  ← 너무 많은 트리
```
Optuna가 과적합 방향으로 수렴한 전형적인 케이스.

In [ ]:
train_pred = best_model.predict(X_train)
train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
print(f"학습 RMSE  : {train_rmse:.1f}")
print(f"테스트 RMSE: {244.8:.1f}")
print(f"차이       : {244.8 - train_rmse:.1f}")

학습 RMSE  : 27.1
테스트 RMSE: 244.8
차이       : 217.7


## ⚠️ 과적합 진단 - 핵심 발견
```
학습 RMSE  : 27.1   ← 훈련 데이터는 거의 완벽하게 맞춤
테스트 RMSE: 244.8  ← 실제 예측은 6배 이상 오차
과적합 갭  : 217.7  ← 심각한 과적합
```
**이 숫자가 08·09번 개선의 출발점이 됨**

## TimeSeriesSplit 교차검증

## 🔄 TimeSeriesSplit 교차검증 - 불안정성 확인
```
Fold 1 RMSE: 265.6
Fold 2 RMSE: 391.8  ← 갑자기 폭등
Fold 3 RMSE: 347.1
Fold 4 RMSE: 156.9  ← 갑자기 낮아짐
Fold 5 RMSE: 271.9

평균: 286.7  표준편차: 80.2
```
**표준편차 80.2의 의미**
- 평균 대비 28% 편차 = 매우 불안정
- Fold마다 학습하는 시간 구간이 달라서 모델 성능이 들쭉날쭉
- 특정 기간(여름/겨울)이 포함되느냐에 따라 RMSE가 크게 변함
- 단순 파라미터 조정으로는 해결 불가 → **피처 자체를 개선해야 한다는 결론**

In [9]:
# 1. TimeSeriesSplit 교차검증
tscv = TimeSeriesSplit(n_splits = 5)

cv_rmses = []

for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train)):
    X_tr = X_train.iloc[train_idx]
    y_tr = y_train.iloc[train_idx]
    X_val = X_train.iloc[val_idx]
    y_val = y_train.iloc[val_idx]

    model = XGBRegressor(**study.best_params, random_state = 42, n_jobs = -1)
    model.fit(X_tr, y_tr)
    pred = model.predict(X_val)

    rmse = np.sqrt(mean_squared_error(y_val, pred))
    cv_rmses.append(rmse)
    print(f"Fold {fold + 1} RMSE: {rmse:.1f}")

print(f"\n평균 CV RMSE: {np.mean(cv_rmses):.1f}")
print(f"표준편차: {np.std(cv_rmses):.1f}")

Fold 1 RMSE: 265.6
Fold 2 RMSE: 391.8
Fold 3 RMSE: 347.1
Fold 4 RMSE: 156.9
Fold 5 RMSE: 271.9

평균 CV RMSE: 286.7
표준편차: 80.2


## CV 기반 Optuna

## 🔍 2차 Optuna - CV 기반 튜닝
**1차 대비 개선 시도**
- TimeSeriesSplit을 Optuna 목적함수 내부에 적용
- `min_child_weight` 범위 5~20으로 소폭 상향

**결과**
```
학습 RMSE  : 49.8
테스트 RMSE: 246.4
차이       : 196.6
```
CV 추가해도 과적합 갭이 줄지 않음 → 파라미터 문제가 아닌 **피처 설계 문제** 확인

In [10]:
# 1. CV 기반 Optuna
def objective_cv(trial):
    params = {
        "n_estimators"     : trial.suggest_int("n_estimators", 100, 1000),
        "max_depth"        : trial.suggest_int("max_depth", 3, 6),
        "learning_rate"    : trial.suggest_float("learning_rate", 0.01, 0.3),
        "subsample"        : trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree" : trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight" : trial.suggest_int("min_child_weight", 5, 20),
        "random_state"     : 42,
        "n_jobs"           : -1
    }

    tscv = TimeSeriesSplit(n_splits = 5)
    cv_rmses = []

    for train_idx, val_idx in tscv.split(X_train):
        X_tr = X_train.iloc[train_idx]
        y_tr = y_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]
        y_val = y_train.iloc[val_idx]

        model = XGBRegressor(**params)
        model.fit(X_tr, y_tr)
        pred = model.predict(X_val)
        cv_rmses.append(np.sqrt(mean_squared_error(y_val, pred)))

    return np.mean(cv_rmses)

# 2. 실행
study_cv = optuna.create_study(direction = "minimize")
study_cv.optimize(objective_cv, n_trials = 200, show_progress_bar = True)

print(f"\n최적 CV RMSE: {study_cv.best_value:.1f}")
print(f"최적 파라미터: {study_cv.best_params}")

# 3. 최적 모델 재학습
best_model_cv = XGBRegressor(**study_cv.best_params, random_state = 42, n_jobs = -1)
best_model_cv.fit(X_train, y_train)
pred_cv = best_model_cv.predict(X_test)

rmse_cv = np.sqrt(mean_squared_error(y_test, pred_cv))
r2_cv = r2_score(y_test, pred_cv)
mape_cv = mean_absolute_percentage_error(y_test, pred_cv) * 100

print(f"\nCV 기반 고도화 모델 성능")
print(f"RMSE : {rmse_cv:.1f}")
print(f"R²   : {r2_cv:.4f}")
print(f"MAPE : {mape_cv:.2f}%")

[I 2026-03-04 12:46:08,137] A new study created in memory with name: no-name-cbc11fc0-089f-4d22-9a11-0f98f49a87b4


  0%|          | 0/200 [00:00<?, ?it/s]

[I 2026-03-04 12:46:10,023] Trial 0 finished with value: 300.67127833320046 and parameters: {'n_estimators': 477, 'max_depth': 6, 'learning_rate': 0.14915621579281169, 'subsample': 0.9892017503961344, 'colsample_bytree': 0.9091486608630439, 'min_child_weight': 15}. Best is trial 0 with value: 300.67127833320046.
[I 2026-03-04 12:46:11,165] Trial 1 finished with value: 290.2664387175588 and parameters: {'n_estimators': 279, 'max_depth': 6, 'learning_rate': 0.10861466650165946, 'subsample': 0.8788391541925908, 'colsample_bytree': 0.9061333384967221, 'min_child_weight': 16}. Best is trial 1 with value: 290.2664387175588.
[I 2026-03-04 12:46:12,327] Trial 2 finished with value: 308.0004043103977 and parameters: {'n_estimators': 503, 'max_depth': 3, 'learning_rate': 0.050983028043828774, 'subsample': 0.6560849769312951, 'colsample_bytree': 0.646307573079266, 'min_child_weight': 20}. Best is trial 1 with value: 290.2664387175588.
[I 2026-03-04 12:46:13,040] Trial 3 finished with value: 309.1

In [11]:
train_pred = best_model_cv.predict(X_train)
train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
print(f"학습 RMSE  : {train_rmse:.1f}")
print(f"테스트 RMSE: {246.4:.1f}")
print(f"차이       : {246.4 - train_rmse:.1f}")

학습 RMSE  : 49.8
테스트 RMSE: 246.4
차이       : 196.6


## 🔍 3차 Optuna - Early Stopping + L1/L2 규제 추가
**추가된 규제 파라미터**
- `reg_alpha` (L1): 불필요한 피처 가중치를 0으로 만들어 희소성 확보
- `reg_lambda` (L2): 모든 가중치를 작게 유지해 과적합 억제
- Early Stopping: 검증 성능이 개선되지 않으면 학습 조기 종료

**결론**: 규제를 추가해도 근본적인 과적합 해소 불가
→ **Rolling 피처 세분화, CDD², 폭염 가중치** 전략으로 방향 전환 (→ 08번)

In [12]:
# ── Early Stopping + 규제 기반 Optuna ─────────────
def objective_final(trial):
    params = {
        "n_estimators"     : trial.suggest_int("n_estimators", 100, 1000),
        "max_depth"        : trial.suggest_int("max_depth", 3, 6),
        "learning_rate"    : trial.suggest_float("learning_rate", 0.01, 0.3),
        "subsample"        : trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree" : trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight" : trial.suggest_int("min_child_weight", 5, 20),
        "reg_alpha"        : trial.suggest_float("reg_alpha", 0, 1),
        "reg_lambda"       : trial.suggest_float("reg_lambda", 0, 1),
        "random_state"     : 42,
        "n_jobs"           : -1,
        "early_stopping_rounds" : 50
    }

    tscv = TimeSeriesSplit(n_splits=5)
    cv_rmses = []

    for train_idx, val_idx in tscv.split(X_train):
        X_tr  = X_train.iloc[train_idx]
        y_tr  = y_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]
        y_val = y_train.iloc[val_idx]

        model = XGBRegressor(**params)
        model.fit(X_tr, y_tr,
                  eval_set = [(X_val, y_val)],
                  verbose = False)
        pred = model.predict(X_val)
        cv_rmses.append(np.sqrt(mean_squared_error(y_val, pred)))

    return np.mean(cv_rmses)

# ── 실행 ──────────────────────────────────────────
study_final = optuna.create_study(direction = "minimize")
study_final.optimize(objective_final, n_trials = 200, show_progress_bar = True)

print(f"\n✅ 최적 CV RMSE  : {study_final.best_value:.1f}")
print(f"✅ 최적 파라미터 : {study_final.best_params}")

# ── 최적 모델 재학습 ──────────────────────────────
best_params_final = study_final.best_params
best_params_final.pop("early_stopping_rounds", None)

best_model_final = XGBRegressor(**best_params_final,
                                 early_stopping_rounds = 50,
                                 random_state = 42, n_jobs = -1)
best_model_final.fit(X_train, y_train,
                     eval_set=[(X_test, y_test)],
                     verbose = False)
pred_final = best_model_final.predict(X_test)

rmse_final = np.sqrt(mean_squared_error(y_test, pred_final))
r2_final   = r2_score(y_test, pred_final)
mape_final = mean_absolute_percentage_error(y_test, pred_final) * 100

print(f"\n📊 최종 고도화 모델 성능")
print(f"RMSE : {rmse_final:.1f}")
print(f"R²   : {r2_final:.4f}")
print(f"MAPE : {mape_final:.2f}%")

# ── 과적합 확인 ───────────────────────────────────
train_pred_final = best_model_final.predict(X_train)
train_rmse_final = np.sqrt(mean_squared_error(y_train, train_pred_final))
print(f"\n📊 과적합 확인")
print(f"학습 RMSE  : {train_rmse_final:.1f}")
print(f"테스트 RMSE: {rmse_final:.1f}")
print(f"차이       : {rmse_final - train_rmse_final:.1f}")

[I 2026-03-04 12:55:51,177] A new study created in memory with name: no-name-76af3439-a0e9-4ec8-baec-a8c9570c74db


  0%|          | 0/200 [00:00<?, ?it/s]

[I 2026-03-04 12:55:52,515] Trial 0 finished with value: 293.98421484600556 and parameters: {'n_estimators': 387, 'max_depth': 6, 'learning_rate': 0.2118970281403343, 'subsample': 0.793121838036464, 'colsample_bytree': 0.6787724181071707, 'min_child_weight': 20, 'reg_alpha': 0.961989232129659, 'reg_lambda': 0.13070401981368407}. Best is trial 0 with value: 293.98421484600556.
[I 2026-03-04 12:55:53,705] Trial 1 finished with value: 274.2078192250701 and parameters: {'n_estimators': 740, 'max_depth': 5, 'learning_rate': 0.21561056876678913, 'subsample': 0.8317038342342639, 'colsample_bytree': 0.8205838464328677, 'min_child_weight': 9, 'reg_alpha': 0.11715103275365257, 'reg_lambda': 0.43493178559959367}. Best is trial 1 with value: 274.2078192250701.
[I 2026-03-04 12:55:55,417] Trial 2 finished with value: 284.63304734473314 and parameters: {'n_estimators': 673, 'max_depth': 4, 'learning_rate': 0.0540740804476169, 'subsample': 0.8802485985458203, 'colsample_bytree': 0.9886990896885786, '

## 🧪 4차 실험 - 최소 피처(FEATURES_V1)로 베이스라인 재확인
**FEATURES_V1 (9개)**
```python
['hour', 'dayofweek', 'month', 'is_weekend', 'is_holiday', 'CDD', 'HDD', 'lag_24', 'lag_168']
```
**이 실험의 목적**
- 환경 피처 없이 시간+냉난방도일+lag만으로 어느 정도 성능이 나오는지 확인
- 피처 추가의 실질적 기여도 측정을 위한 기준점(baseline) 설정
- 결과를 보고 **08번에서 어떤 피처를 추가할지 방향 결정**

In [13]:
# 1. 피쳐 구성

FEATURES_V1 = [
    "hour", "dayofweek", "month",
    "is_weekend", "is_holiday",
    "CDD", "HDD",
    "lag_24", "lag_168"
]

TARGET = "전력사용량(MWh)"

X = df[FEATURES_V1]
y = df[TARGET]

# 2. Train / Test 분리 (연도 기준)

train = df[df["datetime"].dt.year == 2023]
test = df[df["datetime"].dt.year == 2024]

X_train = train[FEATURES_V1]
y_train = train[TARGET]

X_test = test[FEATURES_V1]
y_test = test[TARGET]

# 3. Optuna + TimeSeriesSplit

def objective(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 700),
        "max_depth": trial.suggest_int("max_depth", 2, 4),  # 깊이 제한
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.05),
        "min_child_weight": trial.suggest_int("min_child_weight", 10, 30),
        "subsample": trial.suggest_float("subsample", 0.6, 0.9),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 0.9),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.5, 2.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.5, 3.0),
        "random_state": 42,
        "n_jobs": -1
    }

    tscv = TimeSeriesSplit(n_splits = 5)
    rmse_list = []

    for train_idx, val_idx in tscv.split(X_train):

        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model = XGBRegressor(**params)

        model.fit(
            X_tr, y_tr,
            eval_set = [(X_val, y_val)],
            verbose = False
        )

        pred = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, pred))
        rmse_list.append(rmse)

    return np.mean(rmse_list)

study = optuna.create_study(direction = "minimize")
study.optimize(objective, n_trials = 30)

best_params = study.best_params

# 4. 최종 모델 학습

final_model = XGBRegressor(**best_params)

final_model.fit(X_train, y_train)

# 5. 성능평가

train_pred = final_model.predict(X_train)
test_pred  = final_model.predict(X_test)

train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
test_rmse  = np.sqrt(mean_squared_error(y_test, test_pred))

print(f"📊 Train RMSE : {train_rmse:.2f}")
print(f"📊 Test RMSE  : {test_rmse:.2f}")
print(f"📊 Gap        : {test_rmse - train_rmse:.2f}")

[I 2026-03-04 13:01:46,332] A new study created in memory with name: no-name-9dd9545d-29d4-4c19-b108-84cf3cb84306
[I 2026-03-04 13:01:48,134] Trial 0 finished with value: 330.3786777201039 and parameters: {'n_estimators': 639, 'max_depth': 4, 'learning_rate': 0.016690141083654284, 'min_child_weight': 12, 'subsample': 0.690838730974284, 'colsample_bytree': 0.7401678854178197, 'reg_alpha': 0.8232015753776054, 'reg_lambda': 2.53184784220698}. Best is trial 0 with value: 330.3786777201039.
[I 2026-03-04 13:01:49,470] Trial 1 finished with value: 330.02765422240617 and parameters: {'n_estimators': 461, 'max_depth': 4, 'learning_rate': 0.01553000011389404, 'min_child_weight': 29, 'subsample': 0.6033316248399427, 'colsample_bytree': 0.8587549936782842, 'reg_alpha': 1.3457834194329366, 'reg_lambda': 2.6012299127734706}. Best is trial 1 with value: 330.02765422240617.
[I 2026-03-04 13:01:50,414] Trial 2 finished with value: 349.9581814190676 and parameters: {'n_estimators': 334, 'max_depth': 3,

📊 Train RMSE : 125.89
📊 Test RMSE  : 292.22
📊 Gap        : 166.33
